# Lab 008 — PCA of a Returns Panel, by Hand

**Lesson:** [`0008-linear-algebra-pca.html`](../lessons/0008-linear-algebra-pca.html)

**The one skill:** take a multi-asset returns panel, build its covariance/correlation matrix, eigendecompose it, and read off the principal components — confirming that **PC1 is the market factor** and measuring how much of the panel's risk lives in a few directions.

**Exit criteria:** the EXIT TICKET cell prints cleanly (all CHECK cells pass).

**How this notebook works**

| Cell tag | You do |
|----------|--------|
| **PROVIDED** | Run it. Imports, data, helpers. |
| **TODO** | Fill the `____` blanks. This is where the learning is. |
| **CHECK** | Run it — immediate assertions. Don't edit. |
| **EXIT TICKET** | Final deliverable. Prints your findings. |

**Environment:** Python 3 + `numpy`. Optional real data via `yfinance` (falls back to a simulated factor panel offline). See [`labs/README.md`](./README.md).

### Running on Google Colab?

Colab opens only this single file, so the lab dependencies (numpy, scipy, statsmodels,
yfinance, …) and the course repo are **not** guaranteed to be present. The cell below fixes
that: on Colab it shallow-clones the course repo, installs `requirements-labs.txt`, and
switches into `labs/` so relative paths resolve. **On a local venv or Binder it does nothing —
just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/financial-engineering"):
        !git clone --depth 1 https://github.com/Avistian/financial-engineering.git /content/financial-engineering
    %pip install -q -r /content/financial-engineering/requirements-labs.txt
    os.chdir("/content/financial-engineering/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap (read before coding)

**Centering / standardizing.** Subtract each asset's mean ($\tilde X$). To make assets comparable regardless of their volatility, also divide each column by its std — then the covariance matrix *is* the correlation matrix.

**Covariance matrix** ($d\times d$, symmetric, PSD), from a centered panel $\tilde X$ ($n$ days $\times$ $d$ assets):
$$\Sigma = \frac{1}{n-1}\,\tilde X^{\top}\tilde X.$$

**Eigendecomposition.** $\Sigma = Q\Lambda Q^{\top}$: columns of $Q$ are orthonormal eigenvectors (the principal components), $\Lambda=\mathrm{diag}(\lambda_1\ge\dots\ge\lambda_d)$. Use `np.linalg.eigh` (for symmetric matrices — real eigenvalues, orthonormal eigenvectors).

**Variance explained** by PC $k$: $\lambda_k / \sum_j \lambda_j = \lambda_k / \mathrm{tr}(\Sigma)$. The **trace = total variance = sum of eigenvalues** (a checkable identity).

**Scores (factor returns):** $Z = \tilde X\,Q$; column $k$ is the daily return of the PC-$k$ portfolio.

**Market factor:** on equities, PC1's eigenvector loads with the *same sign on nearly every asset* — the 'everything moves together' direction.

In [ ]:
# PROVIDED — imports and a self-contained returns panel.
# We TRY real data (yfinance); if unavailable/offline we simulate a factor panel where we KNOW
# the truth: every asset = beta * (one common market factor) + idiosyncratic noise, so PC1 must
# recover the market. Either way you get `R` (n_days x d) and asset `names`.
import numpy as np

rng = np.random.default_rng(8)

def simulate_panel(n_days=750, d=12, seed=8):
    g = np.random.default_rng(seed)
    market = g.normal(0.0, 0.010, size=n_days)        # the one common factor
    betas = g.uniform(0.8, 1.2, size=d)               # each asset's exposure to it
    idio = g.normal(0.0, 0.008, size=(n_days, d))     # asset-specific noise
    R = market[:, None] * betas[None, :] + idio
    names = [f"A{j:02d}" for j in range(d)]
    return R, names, market

TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "KO", "PG", "CVX", "BAC", "WFC", "PEP", "GS", "MRK"]
true_market = None
try:
    import yfinance as yf  # optional; needs network
    px = yf.download(TICKERS, period="3y", interval="1d", progress=False, auto_adjust=True)["Close"].dropna()
    R = np.log(px / px.shift(1)).dropna().to_numpy()
    names = list(px.columns)
    print(f"Loaded REAL data: {R.shape[0]} days x {R.shape[1]} assets.")
except Exception as e:
    R, names, true_market = simulate_panel()
    print(f"Using SIMULATED factor panel ({R.shape[0]} days x {R.shape[1]} assets). Reason: {type(e).__name__}.")

n_days, d = R.shape
print("Assets:", names)

### Task 1 — Standardize the panel and build the correlation matrix by hand
**Goal:** center each column (subtract its mean) and scale it to unit std, giving `Xs` (`n_days x d`). Then form the covariance matrix of the standardized data with the matrix formula — this equals the **correlation matrix** `C`. Store `Xs` and `C`.

**Why it matters:** standardizing is the fix for PCA's #1 trap (scale) — otherwise the loudest asset dominates PC1 for no good reason.

**Hint boundary:** `mu = R.mean(axis=0)`, `sd = R.std(axis=0, ddof=1)`, `Xs = (R - mu) / sd`; then `C = (Xs.T @ Xs) / (n_days - 1)`.

In [ ]:
# TODO — standardize, then build the correlation matrix from the matrix formula.
mu = R.mean(axis=0)
sd = R.std(axis=0, ddof=1)
Xs = (R - mu) / sd                       # standardized panel (mean 0, std 1 per column)
C = ____                                 # (Xs.T @ Xs) / (n_days - 1) -> the d x d correlation matrix
print("Correlation matrix shape:", C.shape)
print("Mean off-diagonal correlation: %.3f" % C[~np.eye(d, dtype=bool)].mean())

In [ ]:
# CHECK — do not edit
assert C.shape == (d, d), "C must be d x d"
assert np.allclose(C, C.T, atol=1e-8), "a correlation matrix must be symmetric"
assert np.allclose(np.diag(C), 1.0, atol=1e-6), "standardized-data covariance has unit diagonal"
print("Task 1 OK — correlation matrix built (symmetric, unit diagonal).")

### Task 2 — Eigendecompose and compute variance explained
**Goal:** eigendecompose `C` with `np.linalg.eigh`, sort eigenvalues **descending** (and reorder eigenvectors to match). Store `evals` (length d, descending), `evecs` (`d x d`, column k = PC k), and `var_explained` = `evals / evals.sum()`.

**Why it matters:** these eigenvalues are the variances along each principal direction; their shares are the scree plot.

**Hint boundary:** `w, V = np.linalg.eigh(C)`; `eigh` returns ascending, so `idx = np.argsort(w)[::-1]`; `evals = w[idx]`, `evecs = V[:, idx]`.

In [ ]:
# TODO — eigendecomposition, sorted from most to least variance.
w, V = np.linalg.eigh(C)                  # ascending eigenvalues, orthonormal eigenvectors
idx = np.argsort(w)[::-1]
evals = w[idx]
evecs = ____                              # reorder the eigenvector COLUMNS to match evals
var_explained = evals / evals.sum()
print("Top-5 eigenvalues:", np.round(evals[:5], 3))
print("Variance explained by PC1..PC3: " + ", ".join(f"{v:.1%}" for v in var_explained[:3]))

In [ ]:
# CHECK — do not edit
assert evals.shape == (d,) and evecs.shape == (d, d), "shapes must match the panel"
assert np.all(np.diff(evals) <= 1e-9), "eigenvalues must be sorted descending"
assert np.all(evals > -1e-8), "a covariance/correlation matrix is PSD: no negative eigenvalues"
# Trace identity: sum of eigenvalues == trace == total variance (== d for a correlation matrix).
assert abs(evals.sum() - np.trace(C)) < 1e-6, "sum of eigenvalues must equal the trace"
assert abs(evals.sum() - d) < 1e-6, "for a correlation matrix the trace is d"
print("Task 2 OK — trace identity holds: sum(lambda) = tr(C) = %.2f" % evals.sum())

### Task 3 — Show PC1 is the market factor
**Goal:** compute the **scores** `Z = Xs @ evecs` (`n_days x d`); column 0 is the PC1 portfolio's daily return. Store `pc1_loadings` = `evecs[:, 0]` and `pc1_scores` = `Z[:, 0]`. Then set `same_sign_frac` = the fraction of PC1 loadings sharing the majority sign (a market factor loads the same way on ~all assets).

**Why it matters:** this is the payoff — the data's top eigenvector, with no labels, reconstructs 'the market'.

**Hint boundary:** `Z = Xs @ evecs`; `signs = np.sign(pc1_loadings)`; `same_sign_frac = max((signs > 0).mean(), (signs < 0).mean())`.

In [ ]:
# TODO — project onto the PCs and inspect PC1.
Z = ____                                  # Xs @ evecs  -> daily scores, one column per PC
pc1_loadings = evecs[:, 0]
pc1_scores = Z[:, 0]
signs = np.sign(pc1_loadings)
same_sign_frac = max((signs > 0).mean(), (signs < 0).mean())
print("PC1 loadings (per asset):")
for nm, ld in zip(names, pc1_loadings):
    print(f"  {nm:>6}: {ld:+.3f}")
print(f"Fraction of PC1 loadings with the majority sign: {same_sign_frac:.2f}")
if true_market is not None:
    corr_mkt = np.corrcoef(pc1_scores, true_market[-len(pc1_scores):])[0, 1]
    print(f"|corr(PC1 score, TRUE market factor)| = {abs(corr_mkt):.3f}  (simulated panel)")

In [ ]:
# CHECK — do not edit
assert Z.shape == (n_days, d), "scores must be n_days x d"
# PC scores are uncorrelated by construction: check PC1 vs PC2 correlation is ~0.
c12 = np.corrcoef(Z[:, 0], Z[:, 1])[0, 1]
assert abs(c12) < 1e-6, "principal-component scores must be uncorrelated"
assert same_sign_frac >= 0.75, "PC1 should load with a common sign on most assets (the market factor)"
print("Task 3 OK — PC1 is a common-sign market factor; PC scores are uncorrelated (corr PC1,PC2 = %.1e)." % c12)

### Task 4 — How concentrated is the risk?
**Goal:** compute `cum` = cumulative variance explained (`np.cumsum(var_explained)`) and `n_for_90` = the smallest number of PCs whose cumulative share reaches 90%.

**Why it matters:** this is the dimensionality-reduction headline — 'k factors capture 90% of a d-asset book's risk'.

**Hint boundary:** `cum = np.cumsum(var_explained)`; `n_for_90 = int(np.searchsorted(cum, 0.90) + 1)`.

In [ ]:
# TODO — cumulative variance and the 90% threshold.
cum = np.cumsum(var_explained)
n_for_90 = ____                           # smallest #PCs with cumulative >= 0.90 (1-based)
print("Cumulative variance explained:", ", ".join(f"{c:.1%}" for c in cum))
print(f"PCs needed for >= 90% of variance: {n_for_90} of {d}")

In [ ]:
# CHECK — do not edit
assert 1 <= n_for_90 <= d, "n_for_90 must be a component count"
assert cum[n_for_90 - 1] >= 0.90 - 1e-9, "cumulative share at n_for_90 must reach 90%"
assert abs(cum[-1] - 1.0) < 1e-6, "all components together explain 100% of variance"
print("Task 4 OK — %d of %d components capture >= 90%% of the panel's variance." % (n_for_90, d))

In [ ]:
# EXIT TICKET — paste this output to your teacher.
print("=== PCA of a returns panel ===")
print(f"Panel                          : {n_days} days x {d} assets")
print(f"Mean off-diagonal correlation  : {C[~np.eye(d, dtype=bool)].mean():.3f}")
print(f"Trace identity  sum(lambda)=tr : {evals.sum():.2f} = {np.trace(C):.2f}")
print(f"Variance explained PC1 / PC2   : {var_explained[0]:.1%} / {var_explained[1]:.1%}")
print(f"PC1 majority-sign fraction     : {same_sign_frac:.2f}  (market factor if ~1.0)")
print(f"PCs for >= 90% of variance     : {n_for_90} of {d}")
print()
print("One-sentence takeaway (edit me):")
print("The first principal component of the panel is a same-sign 'market' portfolio that alone")
print(f"explains {var_explained[0]:.0%} of total variance, and just {n_for_90} components capture 90% —")
print("so a d-asset book is really only a handful of independent bets.")

### Stretch (optional, ungraded)
- **Raw vs standardized.** Redo PCA on the *raw* covariance (`np.cov(R, rowvar=False)`) without scaling. Multiply one asset's returns by 100 first and watch PC1 lurch toward that asset — the scale trap, made real.
- **Reconstruction error.** Rebuild the panel from the first `k` PCs (`Xs @ evecs[:, :k] @ evecs[:, :k].T`) and plot reconstruction error vs `k`; confirm the Eckart–Young 'best rank-k' drop matches the discarded eigenvalues' sum.
- **Rolling PC1 share.** Estimate `C` on a rolling 126-day window and plot PC1's variance share over time; spikes flag correlation regimes (crises), the empirical `rho -> 1` effect.
- **RMT noise band (preview, units 082–083).** Simulate a pure-noise panel with the same `n_days, d`; the Marchenko–Pastur upper edge `(1 + sqrt(d/n))**2` marks where real eigenvalues end and noise begins.